In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("IMDB.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [3]:
df.head()

df.drop_duplicates(inplace = True )

In [4]:
df.shape

(49582, 2)

<!-- preprocessing -->

In [5]:
# converting text to LowerCase
df["review"] = df["review"].str.lower()

In [6]:
# Removing the URLs
import re

def remove_url(text):
    new_text = re.sub(r"http\s+","",text)
    return new_text


df["review"] = df["review"].apply(remove_url)

In [7]:
# remove punctuations
def remove_punctuation(text):
    new_text = re.sub(r"[^A-Za-z0-9\s]","",text)
    return new_text

df["review"] = df["review"].apply(remove_punctuation)

In [8]:
# removing HTML Tags
def remove_HTML(text):
    new_text = re.sub(r"<.*?>","",text)
    return new_text

df["review"] = df["review"].apply(remove_HTML)

In [9]:
# removing stopwords
import  nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/aruneshrajawat/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/aruneshrajawat/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/aruneshrajawat/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [10]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [11]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words("english"))

    filtered_words = []

    for word in tokens:
        if word.lower() not in stop_words:
            filtered_words.append(word)

    return " ".join(filtered_words)

In [12]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


In [13]:
# stemming
# Running -> Run
# PorterStemming
from nltk.stem import PorterStemmer

def stemming(text):
    ps = PorterStemmer()
    stemmed_word = []
    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_word.append(stemmed_token)

    return " ".join(stemmed_word)

df["review"] = df["review"].apply(stemming)

In [14]:
df.head()

,review,sentiment
0,one of the other review ha mention that after ...,positive
1,a wonder littl product br br the film techniqu...,positive
2,i thought thi wa a wonder way to spend time on...,positive
3,basic there a famili where a littl boy jake th...,negative
4,petter mattei love in the time of money is a v...,positive


In [15]:
# Encoding 
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])
y = df["sentiment"]

In [16]:
# vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features = 5000)
x = tf.fit_transform(df["review"])

In [17]:
x

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5870928 stored elements and shape (49582, 5000)>

In [18]:
print(x)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5870928 stored elements and shape (49582, 5000)>
  Coords	Values
  (0, 3124)	0.020322456992124902
  (0, 3099)	0.09542070127496334
  (0, 4442)	0.2090340511537307
  (0, 3153)	0.05716473585658308
  (0, 3691)	0.04795004698937513
  (0, 2024)	0.024837052905589747
  (0, 2810)	0.05093608327879004
  (0, 4440)	0.06274935590507338
  (0, 162)	0.03226124598186944
  (0, 4821)	0.07851567951956885
  (0, 2476)	0.04850516109555224
  (0, 3186)	0.43216654344919436
  (0, 1534)	0.10274091540567502
  (0, 4987)	0.052870740488748125
  (0, 443)	0.057113480495623585
  (0, 2178)	0.07161347701650345
  (0, 4457)	0.024445968781803478
  (0, 301)	0.04129740135214664
  (0, 3714)	0.08249959748528456
  (0, 324)	0.07507421173204397
  (0, 4462)	0.042748466069372285
  (0, 2386)	0.12958625075699964
  (0, 1582)	0.05594510691756932
  (0, 4864)	0.04993311987582641
  (0, 2052)	0.04118721045514999
  :	:
  (49581, 2285)	0.09369984967556237
  (49581, 1654)	0.165149520431

In [19]:
# training the RNN
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size = 0.2,random_state = 42)


In [20]:
x_train.shape

(39665, 5000)

In [21]:
x_test.shape

(9917, 5000)

In [22]:
x_train = x_train.toarray()
x_test = x_test.toarray()

In [23]:
import torch
from torch.utils.data import TensorDataset,DataLoader

train_set = TensorDataset(
    torch.from_numpy(x_train).float(),
    torch.from_numpy(y_train.values).float()
)
test_set = TensorDataset(
    torch.from_numpy(x_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [24]:
train_loader = DataLoader(train_set,shuffle = True,batch_size = 64)
test_loader = DataLoader(test_set,shuffle = True,batch_size = 64)


<!-- BUILDING RNN -->

In [25]:
import torch.nn as nn
import torch.optim as optim

class RNN(nn.Module):
    def __init__(self,input_size,hidden_size = 128,num_layers = 1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers


        # RNN layers
        self.rnn = nn.RNN(input_size,hidden_size,num_layers,batch_first = True)


        # fully connected layer
        self.fc = nn.Linear(hidden_size , 1)


    def forward(self,x):
        h0 = torch.zeros(self.num_layers,x.size(0),self.hidden_size)

        out,_ = self.rnn(x,h0)
        # 1st value = hidden state of all time stamps => (batch, seq layer,hidden = size)
        # 2nd value = final hidden state of last time stamps

        out = self.fc(out[:,-1,:])
        return out
    

In [26]:
input_size = x_train.shape[1]
model = RNN(input_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())


In [27]:
# training RNN
epochs = 25

for epoch in range (epochs):
    model.train()
    for Xb,yb in train_loader:
        optimizer.zero_grad()
        Xb = Xb.unsqueeze(1) #skeletal direction

        outputs = model(Xb) #batch_size,1
        outputs = torch.sigmoid(outputs.squeeze()) #batch_size => probalibility

        loss = criterion(outputs,yb)
        loss.backward() #back_prop
        optimizer.step() #weight_update 


    print(f"epoch = {epoch +1}/{epochs} and loss = {loss.item()}")
        

epoch = 1/25 and loss = 0.12535050511360168
epoch = 2/25 and loss = 0.3639654517173767
epoch = 3/25 and loss = 0.21108853816986084
epoch = 4/25 and loss = 0.39419594407081604
epoch = 5/25 and loss = 0.27915138006210327
epoch = 6/25 and loss = 0.2317250519990921
epoch = 7/25 and loss = 0.15717840194702148
epoch = 8/25 and loss = 0.14933201670646667
epoch = 9/25 and loss = 0.17930354177951813
epoch = 10/25 and loss = 0.1880636364221573
epoch = 11/25 and loss = 0.18368856608867645
epoch = 12/25 and loss = 0.1687404215335846
epoch = 13/25 and loss = 0.24999785423278809
epoch = 14/25 and loss = 0.08722412586212158
epoch = 15/25 and loss = 0.05658877268433571
epoch = 16/25 and loss = 0.15469498932361603
epoch = 17/25 and loss = 0.1302616000175476
epoch = 18/25 and loss = 0.2065814733505249
epoch = 19/25 and loss = 0.13702857494354248
epoch = 20/25 and loss = 0.1577092409133911
epoch = 21/25 and loss = 0.3353421688079834
epoch = 22/25 and loss = 0.11396151036024094
epoch = 23/25 and loss = 0.

In [28]:
import pickle
pickle.dump(tf, open("tfidf.pkl", "wb"))

torch.save(model.state_dict(), "model.pth")

In [29]:
model.eval()

with torch.no_grad():
    
    correct_vals = 0
    tot_vals = 0

    for Xb,yb in test_loader:
        
         
         
         Xb = Xb.unsqueeze(1) #skeletal direction
    
         outputs = model(Xb) #batch_size,1
         predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float() #batch_size => probalibility
    
         tot_vals += yb.size(0)
         correct_vals += (predicted == yb).sum().item()


    print (f"accuracy = {correct_vals/tot_vals*100}")
         

         
         
         
        
            

accuracy = 87.05253604920843
